# Stage 7 — forensic evidence aggregation

This notebook combines independent statistical, traffic, temporal, mobility, and social indicators into an investigator-oriented evidence table. It does not train a model and does not claim that an evidence count is proof of fraud.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE=Path('../outputs')
OUTPUT=BASE/'evidence/all_naive_12pct_200sims'
OUTPUT.mkdir(parents=True,exist_ok=True)
stat=pd.read_csv(BASE/'statistical/all_naive_12pct_200sims/sim_statistical_features.csv',dtype={'sim_number':'string'})
traffic=pd.read_csv(BASE/'traffic/all_naive_12pct_200sims/traffic_features.csv',dtype={'sim_number':'string'})
temporal=pd.read_csv(BASE/'temporal/all_naive_12pct_200sims/temporal_features.csv',dtype={'sim_number':'string'})
mobility=pd.read_csv(BASE/'mobility/all_naive_12pct_200sims/mobility_features.csv',dtype={'sim_number':'string'})
social=pd.read_csv(BASE/'social/all_naive_12pct_200sims/social_network_features.csv',dtype={'sim':'string'})
social=social.rename(columns={'sim':'sim_number'})
print(len(stat),len(traffic),len(temporal),len(mobility),len(social))

6992 6992 6992 6992 6992


In [2]:
keys=['sim_number']
features=stat.merge(traffic,on=keys,suffixes=('','_traffic')).merge(temporal,on=keys,suffixes=('','_temporal')).merge(mobility,on=keys,suffixes=('','_mobility')).merge(social,on=keys,suffixes=('','_social'))
features=features.loc[:,~features.columns.duplicated()].copy()
print('merged rows:',len(features),'columns:',len(features.columns))

# Evidence flags are calibrated only against the known-legitimate population.
legit=features.loc[~features.known_fraud]
flags={
 'high_activity':('activity_count', 'high'),
 'high_duration':('total_duration','high'),
 'many_contacts':('unique_contacts','high'),
 'high_outgoing_degree':('outgoing_degree','high'),
 'wide_active_hours':('active_hours','high'),
 'night_activity':('night_activity_ratio','high'),
 'long_activity_span':('activity_span_seconds','high'),
 'mobility_change':('observed_cell_changes','high'),
 'high_contact_repetition':('events_per_contact','high'),
}
for flag,(column,direction) in flags.items():
    q3=legit[column].quantile(.75)
    features[flag]=features[column]>q3 if direction=='high' else features[column]<legit[column].quantile(.25)
    features[flag]=features[flag].fillna(False)
    print(flag,'threshold',q3,'flagged',int(features[flag].sum()))
features['evidence_count']=features[list(flags)].sum(axis=1)
features['evidence_layers']=features[list(flags)].gt(0).sum(axis=1)
features.to_csv(OUTPUT/'forensic_evidence_table.csv',index=False)

merged rows: 6992 columns: 91
high_activity threshold 5.0 flagged 1585
high_duration threshold 279.0 flagged 1775
many_contacts threshold 2.0 flagged 1444
high_outgoing_degree threshold 1.0 flagged 1555
wide_active_hours threshold 3.0 flagged 1493
night_activity threshold 0.0 flagged 659
long_activity_span threshold 21268.0 flagged 1891
mobility_change threshold 4.0 flagged 1585
high_contact_repetition threshold 2.0 flagged 1493


In [3]:
flag_columns=list(flags)
summary=features.groupby('known_fraud')[flag_columns+['evidence_count','evidence_layers']].mean().T.rename(columns={False:'known_legitimate_mean',True:'known_fraudulent_mean'})
summary['difference']=summary.known_fraudulent_mean-summary.known_legitimate_mean
summary.to_csv(OUTPUT/'evidence_group_summary.csv')
ranked=features.sort_values(['evidence_layers','evidence_count','activity_count'],ascending=False)
ranked[['sim_number','known_fraud','evidence_count','evidence_layers']+flag_columns].head(100).to_csv(OUTPUT/'top_evidence_candidates.csv',index=False)
print(summary); print(ranked[['sim_number','known_fraud','evidence_count','evidence_layers']+flag_columns].head(20))

known_fraud              known_legitimate_mean  known_fraudulent_mean  \
high_activity                         0.232740               0.020101   
high_duration                         0.249669               0.396985   
many_contacts                         0.190637               0.748744   
high_outgoing_degree                  0.200501               0.969849   
wide_active_hours                     0.211247               0.291457   
night_activity                        0.077285               0.673367   
long_activity_span                    0.249963               0.969849   
mobility_change                       0.232740               0.020101   
high_contact_repetition               0.219785               0.000000   
evidence_count                        1.864566               4.090452   
evidence_layers                       1.864566               4.090452   

known_fraud              difference  
high_activity             -0.212639  
high_duration              0.147316  
many_cont

Interpretation rule: evidence flags are screening signals. The table is intended to prioritize manual or downstream model review; it is not a final fraud verdict.